# Ранние эксперименты Г1: полный набор метрик и предсказания по людям

| Поле | Значение |
|------|----------|
| **Автор** | Захар Санников (zs) |
| **Дата** | 2026-10-22 |
| **Гипотеза** | Г1 (ЭЭГ) |
| **Цель** | Пересчитать ранние эксперименты **теми же методами**, добавив ROC AUC, accuracy, сбалансированную accuracy, precision, recall, специфичность с 95% ДИ (бутстреп по людям) и сохранив предсказания по людям |
| **Данные** | NMT (`mhmdrdwn/nmt-eeg`), Turkish Epilepsy (`buraktaci/turkish-epilepsy`), Гвинея-Бисау / Нигерия (Zenodo 1252141, качается внутри блокнота) |
| **Модели** | DeepEpilepsy без обучения; REVE (замороженные признаки по каналам + лог. регрессия); DeepEpilepsy, дообученная на NMT (две первые версии) |
| **Пороги** | Не по тесту. Без обучения: 0,5 и порог (Юден / чувствительность 80 %) по предсказаниям на NMT train. Дообучение: по отложенным 15 % NMT train. Признаки REVE: внутри обучающей части каждого разбиения (внутренняя кросс-валидация по людям); перенос — по обучающей стране |
| **Среда** | Kaggle, GPU T4, интернет включён |

Исходные блокноты: `deepepilepsy-turkish`, `deepepilepsy-nmt`, `deepepilepsy-nmt-to-turkish`, `reve-turkish`, `reve-africa`. Методы не менялись.
Метрики пишутся в `early_experiments_metrics.csv`, предсказания — в `early_experiments_per_person.csv` (после каждого эксперимента).
Коды порогов: `thr_0.5` — 0,5; `youden` — индекс Юдена; `sens80` — наибольший порог с чувствительностью ≥ 80 % на данных, где порог выбирался.

## 1. Общее: метрики, пороги, запись результатов

In [ ]:
import os, glob, time, subprocess, importlib.util, warnings
import numpy as np, pandas as pd, torch, torch.nn.functional as F, mne, scipy.io, scipy.signal as ss
from concurrent.futures import ThreadPoolExecutor
from huggingface_hub import hf_hub_download
from scipy.stats import rankdata
from sklearn.metrics import roc_curve, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold, cross_val_predict, train_test_split
mne.set_log_level("ERROR"); warnings.filterwarnings("ignore")
OUT = "/kaggle/working"
torch.manual_seed(0)

def auc_fast(y, s):
    n1 = int(y.sum()); n0 = len(y) - n1
    if n1 == 0 or n0 == 0: return np.nan
    return (rankdata(s)[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)

def _cm(y, p):
    tp = ((p == 1) & (y == 1)).sum(-1); tn = ((p == 0) & (y == 0)).sum(-1)
    fp = ((p == 1) & (y == 0)).sum(-1); fn = ((p == 0) & (y == 1)).sum(-1)
    with np.errstate(all="ignore"):
        rec, spec, prec = tp / (tp + fn), tn / (tn + fp), tp / (tp + fp)
    return np.array([(tp + tn) / (tp + tn + fp + fn), (rec + spec) / 2, prec, rec, spec])

def _stats(y, S, P):
    """ROC AUC и 5 пороговых метрик; при нескольких повторах кросс-валидации — среднее по повторам."""
    return np.concatenate([[np.nanmean([auc_fast(y, s) for s in S])], np.nanmean(_cm(y, P), axis=1)])

def evaluate(y, S, P, B=2000, seed=0):
    rng = np.random.default_rng(seed); n = len(y)
    pt = _stats(y, S, P); bs = []
    for _ in range(B):
        i = rng.integers(0, n, n)                      # бутстреп по людям
        if y[i].min() == y[i].max(): continue
        bs.append(_stats(y[i], S[:, i], P[:, i]))
    lo, hi = np.nanpercentile(np.array(bs), [2.5, 97.5], axis=0)
    return pt, lo, hi

def thr_youden(y, s):
    fpr, tpr, thr = roc_curve(y, s); return float(thr[np.argmax(tpr - fpr)])
def thr_sens(y, s, q=0.8):
    fpr, tpr, thr = roc_curve(y, s); return float(thr[np.argmax(tpr >= q)])

COLS = ["auc", "accuracy", "bal_accuracy", "precision", "recall", "specificity"]
ROWS, PERSON = [], []
def register(exp, data, ids, y, S, preds, thr_vals):
    """S: (повторы, люди); preds: {код порога: (повторы, люди) 0/1}; thr_vals: {код: значение порога}."""
    y = np.asarray(y).astype(int); S = np.atleast_2d(S)
    for name, P in preds.items():
        P = np.atleast_2d(P).astype(int)
        pt, lo, hi = evaluate(y, S, P)
        row = dict(experiment=exp, data=data, threshold=name, threshold_value=thr_vals.get(name, np.nan),
                   n_persons=len(y), n_epilepsy=int(y.sum()))
        for c, a, b, d in zip(COLS, pt, lo, hi):
            row[c] = a; row[c + "_lo"] = b; row[c + "_hi"] = d
        ROWS.append(row)
        print(f"{exp:26s} {data:12s} {name:9s} AUC {pt[0]:.3f} ({lo[0]:.2f}–{hi[0]:.2f}) | acc {pt[1]:.2f} bacc {pt[2]:.2f} "
              f"prec {pt[3]:.2f} rec {pt[4]:.2f} spec {pt[5]:.2f}", flush=True)
    pp = pd.DataFrame(dict(experiment=exp, data=data, id=list(ids), y=y, score=S.mean(0)))
    for name, P in preds.items():
        pp["pred_" + name] = np.atleast_2d(P).mean(0)      # при повторах CV — доля повторов с положительным ответом
    PERSON.append(pp)
    pd.DataFrame(ROWS).to_csv(f"{OUT}/early_experiments_metrics.csv", index=False)
    pd.concat(PERSON).to_csv(f"{OUT}/early_experiments_per_person.csv", index=False)

def fixed_threshold_preds(S, thrs):
    """thrs: {код: порог} -> предсказания (оценка >= порога)."""
    return {k: (np.atleast_2d(S) >= v).astype(int) for k, v in thrs.items()}, dict(thrs)

# самопроверка метрик на искусственных данных
_y = np.r_[np.ones(60), np.zeros(80)].astype(int); _s = np.r_[np.random.default_rng(1).normal(1, 1, 60), np.random.default_rng(2).normal(0, 1, 80)]
assert abs(auc_fast(_y, _s) - roc_auc_score(_y, _s)) < 1e-9
print("метрики готовы; GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "нет")

## 2. DeepEpilepsy: код и веса

In [ ]:
REPO = "/kaggle/working/dl_epilepsy_reeg"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "https://gitlab.com/chum-epilepsy/dl_epilepsy_reeg.git", REPO], check=True)
spec = importlib.util.spec_from_file_location("de_transformers", f"{REPO}/src/epileptology/ml/models/transformers.py")
de = importlib.util.module_from_spec(spec); spec.loader.exec_module(de)
SFREQ, SEG = 200, 30
L = SFREQ * SEG
W_PATH = hf_hub_download("emilelemoine/DeepEpilepsy_v1", "DeepEpilepsy_large_50_30s_augment.pt")
DEV = "cuda" if torch.cuda.is_available() else "cpu"

def new_model():
    m = de.ViT1d(n_class=2, in_channels=19, n_timepoints=L, patch_size=50, tokenizer="conv1d",
                 n_layers=6, n_heads=8, dim_hidden=512, dim_mlp=512, dropout=0.25, attention_dropout=0.25)
    m.load_state_dict(torch.load(W_PATH, map_location="cpu", weights_only=True), strict=True)
    return m.to(DEV)

def scale(x):
    return (x - x.mean(-1, keepdims=True)) / (x.std(-1, keepdims=True) + 1e-20)

def predict(model, Ws, amp=True):
    """Медиана вероятности эпилепсии по окнам для каждой записи."""
    model.eval(); out = []
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16, enabled=amp and DEV == "cuda"):
        for W in Ws:
            p = []
            for b in range(0, len(W), 64):
                x = torch.tensor(scale(W[b:b + 64].astype(np.float32)), dtype=torch.float32).to(DEV)
                p.append(torch.softmax(model(x).float(), 1)[:, 1].cpu().numpy())
            out.append(np.median(np.concatenate(p)))
    return np.array(out)

ZS = new_model()
print("модель готова")

## 3. NMT: чтение (вся обучающая часть — до 8 окон на запись, eval — все окна)

Масштаб сигнала в файлах NMT неверный (СКО ≈0,2 мкВ); обе модели нормируют окна, поэтому на результат это не влияет. Множитель 100 нужен только для хранения в float16.

In [ ]:
ROOT_NMT = glob.glob("/kaggle/input/**/nmt_scalp_eeg_dataset", recursive=True)[0]
lab = pd.read_csv(f"{ROOT_NMT}/Labels.csv")
paths = {os.path.basename(p): p for p in glob.glob(f"{ROOT_NMT}/**/*.edf", recursive=True)}
lab["path"] = lab["recordname"].map(paths)
lab = lab.dropna(subset=["path"]).reset_index(drop=True)
lab["y"] = (lab["label"].str.lower() == "abnormal").astype(int)

# подвыборка для версии «NMT → Турция» (как в исходном блокноте): вся патология + 700 нормальных из train
trn = lab[lab["loc"] == "train"]
SUB_NAMES = set(pd.concat([trn[trn.y == 1], trn[trn.y == 0].sample(700, random_state=0)]).recordname)

def warm(p):
    with open(p, "rb") as f:
        while f.read(1 << 24):
            pass
t0 = time.time()
with ThreadPoolExecutor(32) as ex:
    for i, _ in enumerate(ex.map(warm, lab.path)):
        if (i + 1) % 400 == 0 or i + 1 == len(lab):
            print(f"NMT подкачано {i+1}/{len(lab)} {time.time()-t0:.0f} с", flush=True)

ELECTRODES = ["Fp1", "Fp2", "F3", "F4", "C3", "C4", "P3", "P4", "F7", "F8", "T3", "T4", "T5", "T6", "O1", "O2", "Fz", "Cz", "Pz"]
ALIAS = {"T7": "T3", "T8": "T4", "P7": "T5", "P8": "T6"}
TARGET = [e.upper() for e in ELECTRODES]
def norm(ch):
    c = ch.upper().replace("EEG", "").strip()
    for s in ["-REF", "-AVG", "-LE", "-AR"]:
        c = c.replace(s, "")
    c = c.strip(" -_")
    return ALIAS.get(c, c)

K = 100.0
def nmt_job(i):
    r = lab.iloc[i]
    try:
        raw = mne.io.read_raw_edf(r.path, preload=True)
        if raw.info["sfreq"] != SFREQ:
            raw.resample(SFREQ)
        names = {norm(c): c for c in raw.ch_names}
        X = raw.get_data(picks=[names[t] for t in TARGET]) * 1e6 * K
    except Exception:
        return r.recordname, None
    n = X.shape[1] // L
    if n == 0:
        return r.recordname, None
    W = X[:, :n * L].reshape(19, n, L).transpose(1, 0, 2)
    if r["loc"] == "train" and n > 8:
        W = W[np.sort(np.random.default_rng(i).choice(n, 8, replace=False))]
    return r.recordname, W.astype(np.float16)

NMT_W = {}
t0 = time.time()
with ThreadPoolExecutor(4) as ex:
    for i, (name, W) in enumerate(ex.map(nmt_job, range(len(lab)))):
        if W is not None:
            NMT_W[name] = W
        if (i + 1) % 400 == 0 or i + 1 == len(lab):
            print(f"NMT прочитано {i+1}/{len(lab)} {time.time()-t0:.0f} с", flush=True)
lab = lab[lab.recordname.isin(NMT_W)].reset_index(drop=True)
tr, ev = lab[lab["loc"] == "train"].reset_index(drop=True), lab[lab["loc"] == "eval"].reset_index(drop=True)
ymap = dict(zip(lab.recordname, lab.y))
print(pd.crosstab(lab["loc"], lab["label"]))

## 4. Турецкий набор: чтение

Один проход по файлам даёт три представления: (A) фильтр 0,5–70 Гц + режекция 50 Гц, референт — ухо (основной вариант DeepEpilepsy); (B) без фильтра, средний референт (вариант DeepEpilepsy и вход дообученной модели); (R) фильтр 0,5–45 Гц, 200 Гц, нормировка кусков по 15 с (вход REVE). Куски по 15 с идут подряд (проверено), окно 30 с = два соседних куска.

In [ ]:
ROOT_TR = glob.glob("/kaggle/input/**/Turkish Epilepsy Dataset", recursive=True)[0]
tfiles = [(f, 1) for f in sorted(glob.glob(f"{ROOT_TR}/epilepsy/*.mat"))] + \
         [(f, 0) for f in sorted(glob.glob(f"{ROOT_TR}/normal/*.mat"))]
t0 = time.time()
with ThreadPoolExecutor(16) as ex:
    for i, _ in enumerate(ex.map(warm, [f for f, _ in tfiles])):
        if (i + 1) % 40 == 0 or i + 1 == len(tfiles):
            print(f"Турция подкачано {i+1}/{len(tfiles)} {time.time()-t0:.0f} с", flush=True)

# индексы первых 19 каналов набора в порядке авторов DeepEpilepsy: Fp1 Fp2 F3 F4 C3 C4 P3 P4 F7 F8 T3 T4 T5 T6 O1 O2 Fz Cz Pz
IDX_DE = [0, 1, 2, 3, 5, 6, 8, 9, 13, 14, 15, 16, 17, 18, 11, 12, 4, 7, 10]
SOS_A = ss.butter(4, [0.5, 70], btype="band", fs=500, output="sos"); bn, an = ss.iirnotch(50, 30, fs=500)
SOS_R = ss.butter(4, [0.5, 45], btype="band", fs=500, output="sos")
def pair(X):
    n = len(X) // 2
    return np.concatenate([X[0:2 * n:2], X[1:2 * n:2]], axis=-1)

def tr_job(f):
    d = scipy.io.loadmat(f)["data"]
    X = np.stack([np.stack([d[s, c].ravel() for c in range(19)]) for s in range(d.shape[0])]).astype(np.float64)  # (куски, 19, 7500)
    Xd = X[:, IDX_DE]
    A = pair(ss.resample_poly(ss.filtfilt(bn, an, ss.sosfiltfilt(SOS_A, Xd, axis=-1), axis=-1), 2, 5, axis=-1))
    B = pair(ss.resample_poly(Xd - Xd.mean(1, keepdims=True), 2, 5, axis=-1))
    R = ss.resample_poly(ss.sosfiltfilt(SOS_R, X, axis=-1), 2, 5, axis=-1)
    R = ((R - R.mean(-1, keepdims=True)) / (R.std(-1, keepdims=True) + 1e-8)).astype(np.float32)
    return A.astype(np.float16), B.astype(np.float16), R

TR_A, TR_B, TR_R = [], [], []
t0 = time.time()
with ThreadPoolExecutor(4) as ex:
    for i, (a, b, r) in enumerate(ex.map(tr_job, [f for f, _ in tfiles])):
        TR_A.append(a); TR_B.append(b); TR_R.append(r)
        if (i + 1) % 20 == 0 or i + 1 == len(tfiles):
            print(f"Турция прочитано {i+1}/{len(tfiles)} {time.time()-t0:.0f} с", flush=True)
y_tr = np.array([yy for _, yy in tfiles]); tr_ids = [os.path.basename(f) for f, _ in tfiles]
print("эпилепсия:", int(y_tr.sum()), "| здоровые:", int((1 - y_tr).sum()))

## 5. DeepEpilepsy без обучения

Порог 0,5 — выход модели, как задумано авторами. Порог Юдена и порог на чувствительность 80 % — по предсказаниям той же модели (без дообучения) на NMT train; затем те же значения переносятся на тест. NMT train здесь — до 8 окон на запись, медиана по окнам.

In [ ]:
s_nmt_train = predict(ZS, [NMT_W[n] for n in tr.recordname])
y_nmt_train = tr.y.to_numpy()
THR_ZS = {"thr_0.5": 0.5, "youden": thr_youden(y_nmt_train, s_nmt_train), "sens80": thr_sens(y_nmt_train, s_nmt_train)}
print("пороги без обучения:", THR_ZS, "| ROC AUC на NMT train (только предсказание):", round(auc_fast(y_nmt_train, s_nmt_train), 3))

# NMT eval
s_zs_nmt = predict(ZS, [NMT_W[n] for n in ev.recordname])
P, tv = fixed_threshold_preds(s_zs_nmt, THR_ZS)
register("DE_zs_NMT_eval", "NMT eval", ev.recordname, ev.y, s_zs_nmt, P, tv)

# Турция: основной вариант (A) и средний референт без фильтра (B); в исходном блокноте — fp32
for tag, Ws in [("DE_zs_TR_filter_ear", TR_A), ("DE_zs_TR_meanref", TR_B)]:
    s = predict(ZS, Ws, amp=False)
    P, tv = fixed_threshold_preds(s, THR_ZS)
    register(tag, "Турция", tr_ids, y_tr, s, P, tv)
print("ориентиры: Турция 0,38 (средний референт 0,57), NMT 0,58")

## 6. Дообучение DeepEpilepsy на NMT

Как в исходных блокнотах: AdamW, lr 2·10⁻⁵, weight decay 0,05, классы уравновешены, до 8 окон на запись, без аугментаций; 15 % записей train — выбор эпохи по ROC AUC. Пороги (Юден, чувствительность 80 %) — по предсказаниям лучшей эпохи на этих отложенных 15 %.

In [ ]:
def finetune(names, epochs, tag):
    names = np.array(sorted(names)); yn = np.array([ymap[n] for n in names])
    fit_names, val_names = train_test_split(names, test_size=0.15, stratify=yn, random_state=0)
    Xw = np.concatenate([NMT_W[n] for n in fit_names])
    yw = np.concatenate([np.full(len(NMT_W[n]), ymap[n]) for n in fit_names])
    y_val = np.array([ymap[n] for n in val_names])
    print(f"[{tag}] окон для обучения: {len(yw)} (патология {yw.mean():.2f}) | записей на выбор эпохи: {len(val_names)}", flush=True)
    model = new_model()
    opt = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.05)
    scaler = torch.amp.GradScaler("cuda", enabled=DEV == "cuda")
    w = np.where(yw == 1, 0.5 / (yw == 1).sum(), 0.5 / (yw == 0).sum())
    best_auc, best_state, best_ep = -1, None, 0
    t0 = time.time()
    for ep in range(epochs):
        model.train()
        idx = np.random.default_rng(ep).choice(len(yw), size=len(yw), replace=True, p=w / w.sum())
        losses = []
        for b in range(0, len(idx), 32):
            j = idx[b:b + 32]
            x = torch.tensor(scale(Xw[j].astype(np.float32)), dtype=torch.float32).to(DEV)
            t = torch.tensor(yw[j], dtype=torch.long).to(DEV)
            with torch.autocast("cuda", dtype=torch.float16, enabled=DEV == "cuda"):
                loss = F.cross_entropy(model(x), t)
            opt.zero_grad(); scaler.scale(loss).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); losses.append(loss.item())
        auc = auc_fast(y_val, predict(model, [NMT_W[n] for n in val_names]))
        assert np.isfinite(np.mean(losses)), "потери NaN"
        if auc > best_auc:
            best_auc, best_ep, best_state = auc, ep + 1, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        print(f"[{tag}] эпоха {ep+1}: потери {np.mean(losses):.3f} | ROC AUC на отложенных {auc:.3f} | {time.time()-t0:.0f} с", flush=True)
    model.load_state_dict(best_state)
    del Xw
    s_val = predict(model, [NMT_W[n] for n in val_names])
    thr = {"youden": thr_youden(y_val, s_val), "sens80": thr_sens(y_val, s_val)}
    print(f"[{tag}] выбрана эпоха {best_ep}; пороги по отложенным: {thr}", flush=True)
    return model, thr

# 6.1 всё NMT train, 8 эпох -> NMT eval
m1, thr1 = finetune(tr.recordname.tolist(), 8, "DE_ft_NMT_full")
s1 = predict(m1, [NMT_W[n] for n in ev.recordname])
P, tv = fixed_threshold_preds(s1, thr1)
register("DE_ft_NMT_full", "NMT eval", ev.recordname, ev.y, s1, P, tv)
del m1; torch.cuda.empty_cache()
print("ориентир: NMT eval 0,83")

In [ ]:
# 6.2 подвыборка NMT (вся патология + 700 нормальных), 3 эпохи -> NMT eval и Турция (вход: средний референт, без фильтра)
sub = [n for n in tr.recordname if n in SUB_NAMES]
m2, thr2 = finetune(sub, 3, "DE_ft_NMTsub")
s2 = predict(m2, [NMT_W[n] for n in ev.recordname])
P, tv = fixed_threshold_preds(s2, thr2)
register("DE_ft_NMTsub_to_NMTeval", "NMT eval", ev.recordname, ev.y, s2, P, tv)
s2t = predict(m2, TR_B)
P, tv = fixed_threshold_preds(s2t, thr2)
register("DE_ft_NMTsub_to_TR", "Турция", tr_ids, y_tr, s2t, P, tv)
del m2, ZS, NMT_W, TR_A, TR_B; torch.cuda.empty_cache()
print("ориентиры: NMT eval 0,80; Турция 0,66")

## 7. REVE: признаки «по каналам» + логистическая регрессия

Веса REVE заморожены, один вектор признаков на человека. Кросс-валидация — по людям, 5 частей × 10 повторов (как в исходных блокнотах). В каждом разбиении порог выбирается **только по обучающей части**: оценки для обучающих людей получены внутренней 5-кратной кросс-валидацией (чтобы не брать оценки модели на данных, на которых она обучалась). ROC AUC и метрики усредняются по 10 повторам (в каждом повторе оценки всех людей получены вне обучения), ДИ — бутстреп по людям.

In [ ]:
from transformers import AutoModel
pos_bank = AutoModel.from_pretrained("brain-bzh/reve-positions", trust_remote_code=True)
reve = AutoModel.from_pretrained("brain-bzh/reve-base", trust_remote_code=True).eval().cuda()

def pool_ch(out, C):
    if isinstance(out, (tuple, list)): out = out[0]
    if hasattr(out, "last_hidden_state"): out = out.last_hidden_state
    B, D = out.shape[0], out.shape[-1]
    if out.dim() == 4: per_ch = out.mean(2)
    elif out.dim() == 3 and out.shape[1] % C == 0: per_ch = out.reshape(B, C, -1, D).mean(2)
    else: raise ValueError(f"неожиданная форма выхода {tuple(out.shape)}")
    return per_ch.reshape(B, -1)

def reve_feats(Xr, POS, C, bs):
    """Xr: (окна, C, T) уже нормированные -> средний по окнам вектор (C*512)."""
    ch = []
    with torch.no_grad():
        for b in range(0, len(Xr), bs):
            x = torch.tensor(Xr[b:b + bs], dtype=torch.float32).cuda()
            ch.append(pool_ch(reve(x, POS.expand(x.size(0), -1, -1)), C).cpu())
    return torch.cat(ch).mean(0).numpy()

def make_clf(C):
    return make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000))

def fit_with_thr(Ftr, ytr, Fte, C, k):
    """Пороги — по внутренним (вне обучения) оценкам обучающей части; модель — на всей обучающей части."""
    oof = cross_val_predict(make_clf(C), Ftr, ytr, cv=StratifiedKFold(5, shuffle=True, random_state=k), method="predict_proba")[:, 1]
    thr = {"youden": thr_youden(ytr, oof), "sens80": thr_sens(ytr, oof)}
    return make_clf(C).fit(Ftr, ytr).predict_proba(Fte)[:, 1], thr

def cv_experiment(exp, data, ids, Fm, y, C, n_rep=10):
    n = len(y); S = np.zeros((n_rep, n)); PR = {k: np.zeros((n_rep, n), int) for k in ["youden", "sens80"]}
    tv = {k: [] for k in PR}; fold_auc = []
    for k, (a, b) in enumerate(RepeatedStratifiedKFold(n_splits=5, n_repeats=n_rep, random_state=0).split(Fm, y)):
        s, thr = fit_with_thr(Fm[a], y[a], Fm[b], C, k)
        S[k // 5, b] = s; fold_auc.append(auc_fast(y[b], s))
        for name, t in thr.items():
            PR[name][k // 5, b] = s >= t; tv[name].append(t)
    print(f"{exp}: средний ROC AUC по {len(fold_auc)} разбиениям (как в исходном блокноте) = {np.mean(fold_auc):.3f}")
    register(exp, data, ids, y, S, PR, {k: float(np.mean(v)) for k, v in tv.items()})

def transfer_experiment(exp, data, ids, Ftr, ytr, Fte, yte, C):
    s, thr = fit_with_thr(Ftr, ytr, Fte, C, 0)
    P, tv = fixed_threshold_preds(s, thr)
    register(exp, data, ids, yte, s, P, tv)

# Турция: первые 19 каналов, порядок набора (ухо)
NAMES_TR = ["Fp1","Fp2","F3","F4","Fz","C3","C4","Cz","P3","P4","Pz","O1","O2","F7","F8","T7","T8","P7","P8"]
try:
    POS_TR = pos_bank(NAMES_TR)
except Exception as e:
    print("новые названия не подошли:", e)
    NAMES_TR = ["Fp1","Fp2","F3","F4","Fz","C3","C4","Cz","P3","P4","Pz","O1","O2","F7","F8","T3","T4","T5","T6"]
    POS_TR = pos_bank(NAMES_TR)
POS_TR = POS_TR.cuda()
t0 = time.time()
F_tr = np.stack([reve_feats(R, POS_TR, 19, 16) for R in TR_R])
print("признаки REVE, Турция:", F_tr.shape, f"{time.time()-t0:.0f} с")
assert np.isfinite(F_tr).all()
del TR_R
cv_experiment("REVE_CV_TR", "Турция", tr_ids, F_tr, y_tr, 0.001)
print("ориентир: Турция 0,94")

## 8. REVE на наборе Гвинея-Бисау / Нигерия (Zenodo 1252141, ~0,43 ГБ)

In [ ]:
import urllib.request, zipfile
BASE = "https://zenodo.org/records/1252141/files"; DST = "/kaggle/working/zen"; os.makedirs(DST, exist_ok=True)
for name in ["metadata_nigeria.csv", "metadata_guineabissau.csv", "EEGs_Nigeria.zip", "EEGs_Guinea-Bissau.zip"]:
    p = f"{DST}/{name}"
    if not os.path.exists(p):
        print("качаю", name); urllib.request.urlretrieve(f"{BASE}/{name}?download=1", p)
    if name.endswith(".zip") and not os.path.isdir(p[:-4]):
        zipfile.ZipFile(p).extractall(DST)

ng = pd.read_csv(f"{DST}/metadata_nigeria.csv"); gb = pd.read_csv(f"{DST}/metadata_guineabissau.csv")
rows = []
for _, r in ng.iterrows():
    rows.append(dict(country="NG", subject=f"NG{r['subject.id']}", y=int(str(r["Group"]).lower() == "epilepsy"), path=f"{DST}/EEGs_Nigeria/{r['csv.file']}"))
for _, r in gb.iterrows():
    rows.append(dict(country="GB", subject=f"GB{r['subject.id']}", y=int(str(r["Group"]).lower() == "epilepsy"), path=f"{DST}/EEGs_Guinea-Bissau/signal-{r['subject.id']}.csv.gz"))
rec = pd.DataFrame(rows)
rec = rec[rec.path.map(os.path.exists)]
rec = rec[~rec.path.str.endswith("signal-11-1.csv.gz")]   # в метаданных: «11.1 failed, 11.2 is the right one»
assert rec.groupby("subject").y.nunique().max() == 1
print("записей:", len(rec)); print(rec.groupby(["country", "y"]).subject.nunique().rename("людей").to_frame())

FS, WIN = 128, 10
CH = ["AF3", "AF4", "F3", "F4", "F7", "F8", "FC5", "FC6", "O1", "O2", "P7", "P8", "T7", "T8"]
POS_AF = pos_bank(CH).cuda()
SOS_AF = ss.butter(4, [0.5, 45], btype="band", fs=FS, output="sos")
feats, t0 = [], time.time()
for i, r in enumerate(rec.itertuples()):
    sig = pd.read_csv(r.path, usecols=CH)[CH].to_numpy(dtype=np.float64).T
    sig = ss.sosfiltfilt(SOS_AF, sig, axis=-1)
    Lw = FS * WIN; n = sig.shape[1] // Lw
    X = sig[:, :n * Lw].reshape(14, n, Lw).transpose(1, 0, 2)
    Xr = ss.resample_poly(X, 25, 16, axis=-1)
    Xr = (Xr - Xr.mean(-1, keepdims=True)) / (Xr.std(-1, keepdims=True) + 1e-8)
    feats.append(dict(subject=r.subject, country=r.country, y=r.y, ch=reve_feats(Xr, POS_AF, 14, 32)))
    if (i + 1) % 50 == 0 or i + 1 == len(rec):
        print(f"{i+1}/{len(rec)} {time.time()-t0:.0f} с", flush=True)
df = pd.DataFrame(feats)
subj = df.groupby("subject").agg(country=("country", "first"), y=("y", "first")).reset_index()
F_af = np.stack([np.mean(np.stack(df.loc[df.subject == s, "ch"].tolist()), 0) for s in subj.subject])
assert np.isfinite(F_af).all()
y_af, country, sid = subj.y.to_numpy(), subj.country.to_numpy(), subj.subject.to_numpy()
print("людей:", len(subj), "| эпилепсия:", int(y_af.sum()), "| контроль:", int((1 - y_af).sum()))

In [ ]:
C_AF = 0.01
for c, nm in [("NG", "Нигерия"), ("GB", "Гвинея-Бисау"), ("ALL", "обе страны")]:
    m = np.ones_like(y_af, bool) if c == "ALL" else country == c
    cv_experiment(f"REVE_CV_{c}", nm, sid[m], F_af[m], y_af[m], C_AF)
for a_, b_ in [("NG", "GB"), ("GB", "NG")]:
    a, b = country == a_, country == b_
    transfer_experiment(f"REVE_{a_}_to_{b_}", f"{b_} (обуч. {a_})", sid[b], F_af[a], y_af[a], F_af[b], y_af[b], C_AF)
print("ориентиры: внутри стран 0,93–0,96 (обе вместе 0,93); перенос 0,52 (NG→GB), 0,60 (GB→NG)")

## 9. Итоговая таблица

In [ ]:
m = pd.read_csv(f"{OUT}/early_experiments_metrics.csv")
fmt = lambda r, c: f"{r[c]:.2f} ({r[c+'_lo']:.2f}–{r[c+'_hi']:.2f})"
show = pd.DataFrame({"эксперимент": m.experiment, "данные": m.data, "порог": m.threshold + " = " + m.threshold_value.round(3).astype(str),
                     "ROC AUC": m.apply(lambda r: fmt(r, "auc"), axis=1), "accuracy": m.apply(lambda r: fmt(r, "accuracy"), axis=1),
                     "сбал. acc": m.apply(lambda r: fmt(r, "bal_accuracy"), axis=1), "precision": m.apply(lambda r: fmt(r, "precision"), axis=1),
                     "recall": m.apply(lambda r: fmt(r, "recall"), axis=1), "специфичн.": m.apply(lambda r: fmt(r, "specificity"), axis=1)})
pd.set_option("display.width", 250, "display.max_columns", 20, "display.max_colwidth", 40)
print(show.to_string(index=False))
assert np.isfinite(m.auc).all()
pp = pd.read_csv(f"{OUT}/early_experiments_per_person.csv"); print(pp.groupby("experiment").size())

## 10. Результаты запуска на Kaggle (2026-10-22)

Таблица собрана из `early_experiments_metrics.csv` (пороги Юдена и 0,5; пороги на чувствительность 80 % — в csv). Выводы ячеек в этот файл не вставлены: Kaggle в пакетном режиме не вернул журнал запуска, поэтому выводы по эпохам здесь не приведены.

| эксперимент | данные | порог | ROC AUC | accuracy | сбал. acc | precision | recall | специфичность |
|---|---|---|---|---|---|---|---|---|
| DE_zs_NMT_eval | NMT eval | thr_0.5 = 0.5 | 0.58 (0.49–0.66) | 0.51 (0.43–0.57) | 0.49 (0.48–0.50) | 0.00 (0.00–0.00) | 0.00 (0.00–0.00) | 0.99 (0.97–1.00) |
| DE_zs_NMT_eval | NMT eval | youden = 0.000214 | 0.58 (0.49–0.66) | 0.49 (0.41–0.56) | 0.50 (0.43–0.55) | 0.49 (0.40–0.57) | 0.77 (0.68–0.85) | 0.22 (0.14–0.31) |
| DE_zs_TR_filter_ear | Турция | thr_0.5 = 0.5 | 0.38 (0.28–0.48) | 0.60 (0.50–0.68) | 0.51 (0.49–0.55) | 0.67 (0.00–1.00) | 0.04 (0.00–0.10) | 0.99 (0.95–1.00) |
| DE_zs_TR_filter_ear | Турция | youden = 0.000214 | 0.38 (0.28–0.48) | 0.40 (0.31–0.48) | 0.47 (0.43–0.51) | 0.40 (0.32–0.49) | 0.92 (0.83–0.98) | 0.03 (0.00–0.07) |
| DE_zs_TR_meanref | Турция | thr_0.5 = 0.5 | 0.56 (0.46–0.67) | 0.60 (0.52–0.69) | 0.52 (0.50–0.55) | 1.00 (1.00–1.00) | 0.04 (0.00–0.11) | 1.00 (1.00–1.00) |
| DE_zs_TR_meanref | Турция | youden = 0.000214 | 0.56 (0.46–0.67) | 0.53 (0.44–0.62) | 0.53 (0.44–0.62) | 0.44 (0.32–0.56) | 0.54 (0.40–0.68) | 0.52 (0.41–0.63) |
| DE_ft_NMT_full | NMT eval | youden = 0.0115 | 0.86 (0.81–0.91) | 0.78 (0.72–0.84) | 0.78 (0.73–0.84) | 0.75 (0.66–0.83) | 0.83 (0.75–0.90) | 0.73 (0.64–0.82) |
| DE_ft_NMTsub_to_NMTeval | NMT eval | youden = 0.435 | 0.82 (0.76–0.88) | 0.74 (0.67–0.80) | 0.74 (0.67–0.80) | 0.77 (0.67–0.86) | 0.67 (0.57–0.76) | 0.81 (0.73–0.88) |
| DE_ft_NMTsub_to_TR | Турция | youden = 0.435 | 0.68 (0.58–0.77) | 0.57 (0.48–0.65) | 0.59 (0.50–0.68) | 0.49 (0.38–0.59) | 0.72 (0.59–0.84) | 0.46 (0.35–0.58) |
| REVE_CV_TR | Турция | youden = 0.365 | 0.93 (0.89–0.97) | 0.85 (0.79–0.90) | 0.84 (0.79–0.90) | 0.81 (0.71–0.90) | 0.83 (0.74–0.91) | 0.86 (0.78–0.92) |
| REVE_CV_NG | Нигерия | youden = 0.601 | 0.96 (0.93–0.98) | 0.91 (0.88–0.94) | 0.91 (0.88–0.94) | 0.93 (0.89–0.97) | 0.91 (0.87–0.95) | 0.91 (0.85–0.96) |
| REVE_CV_GB | Гвинея-Бисау | youden = 0.404 | 0.92 (0.87–0.96) | 0.81 (0.75–0.88) | 0.81 (0.75–0.87) | 0.81 (0.71–0.91) | 0.84 (0.77–0.91) | 0.78 (0.68–0.89) |
| REVE_CV_ALL | обе страны | youden = 0.571 | 0.93 (0.90–0.95) | 0.86 (0.83–0.89) | 0.86 (0.82–0.89) | 0.89 (0.84–0.93) | 0.86 (0.82–0.90) | 0.86 (0.80–0.90) |
| REVE_NG_to_GB | GB (обуч. NG) | youden = 0.636 | 0.52 (0.40–0.64) | 0.49 (0.39–0.60) | 0.50 (0.41–0.60) | 0.53 (0.36–0.69) | 0.37 (0.24–0.51) | 0.63 (0.49–0.77) |
| REVE_GB_to_NG | NG (обуч. GB) | youden = 0.309 | 0.60 (0.52–0.68) | 0.61 (0.55–0.68) | 0.59 (0.52–0.65) | 0.65 (0.57–0.73) | 0.73 (0.65–0.80) | 0.45 (0.34–0.55) |

Совпадение с прежними ROC AUC: DE без обучения 0,38 / 0,57 / NMT 0,58; REVE Турция 0,94 (0,935), внутри стран 0,92–0,96, перенос 0,52 / 0,60; DE на NMT 0,86 (раньше 0,83, в пределах ДИ; обучение стохастическое), NMT→Турция 0,68 (раньше 0,66).